In [ ]:
!pip install pandas ipywidgets textblob scikit-learn tqdm numpy openpyxl matplotlib sentence-transformers

  Using cached cycler-0.12.1-py3-none-any.whl.metadata (3.8 kB)
  Using cached kiwisolver-1.4.8-cp311-cp311-win_amd64.whl.metadata (6.3 kB)
   ---------------------------------------- 0.0/8.1 MB ? eta -:--:--
   -- ------------------------------------- 0.5/8.1 MB 2.8 MB/s eta 0:00:03
   ----- ---------------------------------- 1.0/8.1 MB 2.8 MB/s eta 0:00:03
   ------- -------------------------------- 1.6/8.1 MB 2.8 MB/s eta 0:00:03
   ---------- ----------------------------- 2.1/8.1 MB 2.9 MB/s eta 0:00:03
   ------------- -------------------------- 2.6/8.1 MB 2.7 MB/s eta 0:00:03
   --------------- ------------------------ 3.1/8.1 MB 2.7 MB/s eta 0:00:02
   ------------------- -------------------- 3.9/8.1 MB 2.7 MB/s eta 0:00:02
   ---------------------- ----------------- 4.5/8.1 MB 2.7 MB/s eta 0:00:02
   ------------------------ --------------- 5.0/8.1 MB 2.7 MB/s eta 0:00:02
   --------------------------- ------------ 5.5/8.1 MB 2.8 MB/s eta 0:00:01
   ----------------------------

In [36]:
import pandas as pd

df = pd.read_excel("behaviour_simulation_train.xlsx", parse_dates = ['date']) #add path to the dataset

# Data cleaning
df['content'] = df['content'].astype(str).str.strip()
df['username'] = df['username'].astype(str).str.strip()
df['media'] = df['media'].astype(str).str.strip()

# Feature engineering
df['hour'] = df['date'].dt.hour
df['day_of_week'] = df['date'].dt.day_name()
df['month'] = df['date'].dt.month
df['year'] = df['date'].dt.year

media_dict = {'P':'Photo', 'V':'Video', 'G':'Gif'}
df['media_type'] = df['media'].str[1].map(media_dict)

df['word_count'] = df['content'].str.split().str.len()
df['char_count'] = df['content'].str.len()

In [38]:
# Sentiment analysis using TextBlob, adding polarity and subjectivity as features

from textblob import TextBlob
from tqdm.notebook import tqdm

def analyze_sentiments(texts):
    polarities = []
    subjectivities = []
    
    # Use tqdm for progress tracking
    for text in tqdm(texts, desc="Analyzing texts"):
        analysis = TextBlob(text)
        polarities.append(analysis.sentiment.polarity)
        subjectivities.append(analysis.sentiment.subjectivity)
    
    return polarities, subjectivities

df['sentiment_polarity'], df['sentiment_subjectivity'] = analyze_sentiments(df['content'])

Analyzing texts:   0%|          | 0/300000 [00:00<?, ?it/s]

In [39]:
df.info()
display(df.isnull().sum())
display(df.iloc[0:10])

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 300000 entries, 0 to 299999
Data columns (total 16 columns):
 #   Column                  Non-Null Count   Dtype         
---  ------                  --------------   -----         
 0   id                      300000 non-null  int64         
 1   date                    300000 non-null  datetime64[ns]
 2   likes                   300000 non-null  int64         
 3   content                 300000 non-null  object        
 4   username                300000 non-null  object        
 5   media                   300000 non-null  object        
 6   inferred company        300000 non-null  object        
 7   hour                    300000 non-null  int32         
 8   day_of_week             300000 non-null  object        
 9   month                   300000 non-null  int32         
 10  year                    300000 non-null  int32         
 11  media_type              300000 non-null  object        
 12  word_count              300000

id                        0
date                      0
likes                     0
content                   0
username                  0
media                     0
inferred company          0
hour                      0
day_of_week               0
month                     0
year                      0
media_type                0
word_count                0
char_count                0
sentiment_polarity        0
sentiment_subjectivity    0
dtype: int64

,id,date,likes,content,username,media,inferred company,hour,day_of_week,month,year,media_type,word_count,char_count,sentiment_polarity,sentiment_subjectivity
0,1,2020-12-12 00:47:00,1,"Spend your weekend morning with a Ham, Egg, an...",TimHortonsPH,[Photo(previewUrl='https://pbs.twimg.com/media...,tim hortons,0,Saturday,12,2020,Photo,29,181,0.175000,0.325000
1,2,2018-06-30 10:04:20,2750,Watch rapper <mention> freestyle for over an H...,IndyMusic,[Photo(previewUrl='https://pbs.twimg.com/media...,independent,10,Saturday,6,2018,Photo,10,73,0.000000,0.000000
2,3,2020-09-29 19:47:28,57,Canadian Armenian community demands ban on mil...,CBCCanada,[Photo(previewUrl='https://pbs.twimg.com/media...,cbc,19,Tuesday,9,2020,Photo,14,104,-0.100000,0.100000
3,4,2020-10-01 11:40:09,152,"1st in Europe to be devastated by COVID-19, It...",MKWilliamsRome,[Photo(previewUrl='https://pbs.twimg.com/media...,williams,11,Thursday,10,2020,Photo,22,140,0.500000,0.900000
4,5,2018-10-19 14:30:46,41,Congratulations to Pauletha Butts of <mention>...,BGISD,[Photo(previewUrl='https://pbs.twimg.com/media...,independent,14,Friday,10,2018,Photo,26,199,0.062500,0.083333
5,6,2020-11-15 16:01:08,525,An 85-year-old primary school in Shanghai has ...,cnni,[Video(thumbnailUrl='https://pbs.twimg.com/amp...,cnn,16,Sunday,11,2020,Video,28,181,0.268182,0.477273
6,7,2019-10-24 10:51:03,0,LASU Celebrates New Dawn Of Unbroken Peace As ...,IndependentNGR,[Photo(previewUrl='https://pbs.twimg.com/media...,independent,10,Thursday,10,2019,Photo,16,107,0.136364,0.454545
7,8,2018-07-17 22:04:26,3,Next week CNCF will be publishing a series of ...,CiscoCloud,[Video(thumbnailUrl='https://pbs.twimg.com/amp...,cisco,22,Tuesday,7,2018,Video,35,241,0.266667,0.500000
8,9,2019-03-27 12:18:01,572,A 95-year-old World War II veteran says he was...,cnni,[Photo(previewUrl='https://pbs.twimg.com/media...,cnn,12,Wednesday,3,2019,Photo,36,224,-0.287879,0.484848
9,10,2020-08-01 05:24:03,127,"Nicholas Hoult, Charlize Theron, and Aisha Tyl...",GettyVIP,[Photo(previewUrl='https://pbs.twimg.com/media...,getty images,5,Saturday,8,2020,Photo,42,274,-0.062500,0.750000


In [43]:
# one-hot encoding categorical features
# and creating average likes per username and company

from sklearn.model_selection import train_test_split
import numpy as np

X = df
y = df['likes']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
encoder.fit(X_train[['media_type']])
media_type_encoded_train = encoder.transform(X_train[['media_type']])
media_type_encoded_test = encoder.transform(X_test[['media_type']])

encoder.fit(X_train[['inferred company']])
company_encoded_train = encoder.transform(X_train[['inferred company']])
company_encoded_test = encoder.transform(X_test[['inferred company']])

encoder.fit(X_train[['hour']])
hour_encoded_train = encoder.transform(X_train[['hour']])
hour_encoded_test = encoder.transform(X_test[['hour']])

encoder.fit(X_train[['username']])
username_encoded_train = encoder.transform(X_train[['username']])
username_encoded_test = encoder.transform(X_test[['username']])

username_avg_likes_train = X_train.groupby('username')['likes'].mean().reindex(X_train['username']).values
username_avg_likes_test = X_test.groupby('username')['likes'].mean().reindex(X_test['username']).values

company_avg_likes_train = X_train.groupby('inferred company')['likes'].mean().reindex(X_train['inferred company']).values
company_avg_likes_test = X_test.groupby('inferred company')['likes'].mean().reindex(X_test['inferred company']).values

In [ ]:
# RANDOM FOREST REGRESSOR
# 
# WITH FEATURES:
X_train_final = np.hstack([
    X_train['char_count'].values.reshape(-1, 1),
    X_train['sentiment_polarity'].values.reshape(-1, 1), 
    X_train['sentiment_subjectivity'].values.reshape(-1, 1),
    media_type_encoded_train, 
    hour_encoded_train,
])
X_test_final = np.hstack([
    X_test['char_count'].values.reshape(-1, 1),
    X_test['sentiment_polarity'].values.reshape(-1, 1), 
    X_test['sentiment_subjectivity'].values.reshape(-1, 1),
    media_type_encoded_test, 
    hour_encoded_test,
])

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error

model = RandomForestRegressor(
    random_state=42, 
)
model.fit(X_train_encoded, y_train)
preds = model.predict(X_test_encoded)
rmse = np.sqrt(mean_squared_error(y_test, preds))
print("RMSE:", rmse)

In [ ]:
# XGBoost Regressor
#
# WITH FEATURES:
X_train_final = np.hstack([
    X_train['char_count'].values.reshape(-1, 1),
    X_train['sentiment_polarity'].values.reshape(-1, 1), 
    X_train['sentiment_subjectivity'].values.reshape(-1, 1),
    media_type_encoded_train, 
    hour_encoded_train,
])
X_test_final = np.hstack([
    X_test['char_count'].values.reshape(-1, 1),
    X_test['sentiment_polarity'].values.reshape(-1, 1), 
    X_test['sentiment_subjectivity'].values.reshape(-1, 1),
    media_type_encoded_test, 
    hour_encoded_test,
])

from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error

xgb_model = XGBRegressor(
    device='cuda',
    early_stopping_rounds=10,  # to prevent overfitting
    eval_metric='rmse',       
    n_estimators=1000          
)
xgb_model.fit(
    X_train_final, y_train,
    eval_set=[(X_test_final, y_test)],
    verbose=True
)
preds = xgb_model.predict(X_test_final)
rmse = np.sqrt(mean_squared_error(y_test, preds))
print("RMSE:", rmse)

[[7.20000000e+01 2.00000000e-01 1.00000000e-01 ... 0.00000000e+00
  1.00000000e+00 0.00000000e+00]
 [1.38000000e+02 4.16666667e-01 7.12500000e-01 ... 0.00000000e+00
  1.00000000e+00 0.00000000e+00]
 [6.30000000e+01 0.00000000e+00 0.00000000e+00 ... 0.00000000e+00
  1.00000000e+00 0.00000000e+00]
 ...
 [1.91000000e+02 0.00000000e+00 0.00000000e+00 ... 0.00000000e+00
  1.00000000e+00 0.00000000e+00]
 [1.29000000e+02 6.90476190e-02 6.57142857e-01 ... 0.00000000e+00
  1.00000000e+00 0.00000000e+00]
 [1.56000000e+02 4.00000000e-01 3.75000000e-01 ... 0.00000000e+00
  1.00000000e+00 0.00000000e+00]]
(240000, 250)
Fitted 1/100 trees
RMSE: 6501.609497453018 difference: 1570.6094974530179
Fitted 2/100 trees
RMSE: 5684.48028764998 difference: -817.1292098030381
Fitted 3/100 trees
RMSE: 5470.929753743674 difference: -213.55053390630565
Fitted 4/100 trees
RMSE: 5413.836016620225 difference: -57.0937371234495
Fitted 5/100 trees
RMSE: 5279.165448654162 difference: -134.67056796606266
Fitted 6/100 tre

KeyboardInterrupt: 

In [4]:
'''
Features:
- username (average likes over training set)
- inferred company (average likes over training set)
- hour (one-hot encoded)
- media_type (one-hot encoded)
- char_count
- sentiment_polarity
- sentiment_subjectivity
'''

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error
import joblib
import numpy as np
from sklearn.preprocessing import OneHotEncoder

y = df['likes']

X_train, X_test, y_train, y_test = train_test_split(df, y, test_size=0.2, random_state=42)

# using log(likes) as target variable
y_train = np.log1p(y_train)

encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
encoder.fit(df[['media_type', 'hour']])

X_train_final = encoder.transform(X_train[['media_type', 'hour']])

X_test_final = encoder.transform(X_test[['media_type', 'hour']])


username_avg_likes_train = X_train.groupby('username')['likes'].transform('mean').values.reshape(-1,1)
company_avg_likes_train = X_train.groupby('inferred company')['likes'].transform('mean').values.reshape(-1,1)

# username_avg_likes for x_test will be the username_avg_likes from the training set if username is in training set\
# else it will be 0 

username_avg_likes_test = X_test['username'].map(X_train.groupby('username')['likes'].mean()).fillna(0).values.reshape(-1,1)
company_avg_likes_test = X_test['inferred company'].map(X_train.groupby('inferred company')['likes'].mean()).fillna(0).values.reshape(-1,1)


X_train_final = np.hstack([
    X_train['char_count'].values.reshape(-1,1),
    X_train['word_count'].values.reshape(-1,1), 
    X_train['sentiment_polarity'].values.reshape(-1,1), 
    X_train['sentiment_subjectivity'].values.reshape(-1,1),
    X_train_final,  # encoded media_type
    username_avg_likes_train,
    company_avg_likes_train
])

X_test_final = np.hstack([
    X_test['char_count'].values.reshape(-1,1),
    X_test['word_count'].values.reshape(-1,1),
    X_test['sentiment_polarity'].values.reshape(-1,1), 
    X_test['sentiment_subjectivity'].values.reshape(-1,1),
    X_test_final,  # encoded media_type
    username_avg_likes_test,
    company_avg_likes_test
])

model = RandomForestRegressor(random_state=21)
model.fit(X_train_final, y_train)
preds = model.predict(X_test_final)
from sklearn.metrics import mean_squared_error

preds = np.expm1(preds)
rmse = np.sqrt(mean_squared_error(y_test, preds))
print("RMSE:", rmse)


RMSE: 4391.109625032822


In [ ]:
'''
Features:
'''

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error
import joblib
import numpy as np
from sklearn.preprocessing import OneHotEncoder

y = df_merged['likes']

X_train, X_test, y_train, y_test = train_test_split(df_merged, y, test_size=0.2, random_state=42)

# using log(likes) as target variable
y_train = np.log1p(y_train)

encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
encoder.fit(df[['media_type', 'hour']])

X_train_final = encoder.transform(X_train[['media_type', 'hour']])

X_test_final = encoder.transform(X_test[['media_type', 'hour']])

# THERE ARE TWO STRATEGIES: MEAN STRATEGY AND 0 STRATEGY

username_avg_likes_train = X_train.groupby('username')['likes'].transform('mean').values.reshape(-1,1)
company_avg_likes_train = X_train.groupby('inferred company')['likes'].transform('mean').values.reshape(-1,1)

# username_avg_likes for x_test will be the username_avg_likes from the training set if username is in training set\
# else it will be 0 

username_avg_likes_test = X_test['username'].map(X_train.groupby('username')['likes'].mean()).fillna(0).values.reshape(-1,1)
company_avg_likes_test = X_test['inferred company'].map(X_train.groupby('inferred company')['likes'].mean()).fillna(0).values.reshape(-1,1)

# no media_type, polarity, subjectivity, word_count, username_avg, company_avg
X_train_final = np.hstack([
    X_train['char_count'].values.reshape(-1,1),
    X_train['followers'].values.reshape(-1,1),
    X_train['following'].values.reshape(-1,1),
    X_train['statusesCount'].values.reshape(-1,1),
    X_train['favouritesCount'].values.reshape(-1,1),
    X_train['mediaCount'].values.reshape(-1,1)
])

X_test_final = np.hstack([
    X_test['char_count'].values.reshape(-1,1),
    X_test['followers'].values.reshape(-1,1),
    X_test['following'].values.reshape(-1,1),
    X_test['statusesCount'].values.reshape(-1,1),
    X_test['favouritesCount'].values.reshape(-1,1),
    X_test['mediaCount'].values.reshape(-1,1)
])

model = RandomForestRegressor(random_state=21)
model.fit(X_train_final, y_train)
preds = model.predict(X_test_final)
from sklearn.metrics import mean_squared_error

preds = np.expm1(preds)
rmse = np.sqrt(mean_squared_error(y_test, preds))
print("RMSE:", rmse)


RMSE: 4565.388493826037


In [11]:
usernames_data = pd.read_csv("usernames_data_x.csv", parse_dates=['createdAt'])
usernames_data.info()
display(usernames_data.isnull().sum())
display(usernames_data.head())


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2249 entries, 0 to 2248
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype              
---  ------           --------------  -----              
 0   userName         2249 non-null   object             
 1   id               2249 non-null   float64            
 2   name             2249 non-null   object             
 3   followers        2249 non-null   int64              
 4   following        2249 non-null   int64              
 5   isBlueVerified   2249 non-null   bool               
 6   verifiedType     604 non-null    object             
 7   statusesCount    2249 non-null   int64              
 8   mediaCount       2249 non-null   int64              
 9   favouritesCount  2249 non-null   int64              
 10  createdAt        2249 non-null   datetime64[ns, UTC]
 11  location         1788 non-null   object             
 12  description      2184 non-null   object             
 13  canDm            2

userName              0
id                    0
name                  0
followers             0
following             0
isBlueVerified        0
verifiedType       1645
statusesCount         0
mediaCount            0
favouritesCount       0
createdAt             0
location            461
description          65
canDm                 0
dtype: int64

,userName,id,name,followers,following,isBlueVerified,verifiedType,statusesCount,mediaCount,favouritesCount,createdAt,location,description,canDm
0,TimHortonsPH,7.944183e+17,Tim Hortons Philippines,2041,9,False,NaN,931,687,954,2016-11-04 05:57:45+00:00,Republic of the Philippines,Welcome to the official account of Tim Hortons...,False
1,IndyMusic,1.730888e+07,Independent Music,31033,557,False,NaN,31733,4853,104,2008-11-11 12:43:57+00:00,London,"Music news, reviews, and feature stories from ...",False
2,CBCCanada,1.896579e+07,CBC Canadian News,245542,5,False,NaN,93717,53557,11,2009-01-14 03:42:38+00:00,Canada,"Canadian news and features, and the Politics B...",False
3,MKWilliamsRome,7.549140e+08,Megan Williams,6455,1007,False,NaN,5739,1294,10180,2012-08-13 10:53:48+00:00,NaN,"Report on radio & TV & write on Italy, Europe ...",False
4,BGISD,2.206200e+07,Bowling Green Independent Schools,7813,193,True,NaN,7397,2347,5311,2009-02-26 21:06:29+00:00,"Bowling Green, Kentucky","Every day, accomplishments of students and emp...",False


In [1]:
df_merged = pd.merge(df, usernames_data, left_on='username', right_on='userName', how='left')
print(df_merged.shape)
print(df_merged.info())
display(df_merged.isnull().sum())
print(df_merged.head())

NameError: name 'pd' is not defined

In [165]:
display(df_merged.isnull().sum())

id_x                           0
date                           0
likes                          0
content                        0
username                       0
media                          0
inferred company               0
hour                           0
day_of_week                    0
month                          0
year                           0
media_type                     0
word_count                     0
char_count                     0
sentiment_polarity             0
sentiment_subjectivity         0
userName                   11656
id_y                       11656
name                       11656
followers                  11656
following                  11656
isBlueVerified             11656
verifiedType              178564
statusesCount              11656
mediaCount                 11656
favouritesCount            11656
createdAt                  11656
location                   68585
description                15284
canDm                      11656
dtype: int

In [149]:
df_merged.to_csv('300,000_tweets.csv', index=False)

In [120]:
import os
size_bytes = os.path.getsize('behaviour_simulation_train.xlsx')
print(f"Size of '300,000_tweets.csv': {size_bytes / (1024 * 1024)} MB")

Size of '300,000_tweets.csv': 54.112149238586426 MB


In [28]:
df = pd.read_csv('300,000_tweets.csv', parse_dates=['date', 'createdAt'])
print(df.isnull().sum())

id_x                               0
date                               0
likes                              0
content                            0
username                           0
media                              0
inferred company                   0
hour                               0
day_of_week                        0
month                              0
year                               0
media_type                         0
word_count                         0
char_count                         0
sentiment_polarity                 0
sentiment_subjectivity             0
userName                       11656
id_y                           11656
name                           11656
followers                      11656
following                      11656
isBlueVerified                 11656
verifiedType                  178564
statusesCount                  11656
mediaCount                     11656
favouritesCount                11656
createdAt                      11656
l

In [54]:
!pip install sentence-transformers
!pip install transformers
!pip install tf-keras


   ---------------------------------------- 0.0/1.7 MB ? eta -:--:--
   ------------ --------------------------- 0.5/1.7 MB 3.3 MB/s eta 0:00:01
   ------------------------------ --------- 1.3/1.7 MB 3.5 MB/s eta 0:00:01
   ---------------------------------------- 1.7/1.7 MB 3.6 MB/s eta 0:00:00


In [44]:
from sentence_transformers import SentenceTransformer
model = SentenceTransformer('all-MiniLM-L6-v2')

embeddings = model.encode(df['content'].tolist(), show_progress_bar=True, convert_to_numpy=True)
print(embeddings.shape)
print(embeddings)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

c:\Users\swaya\python\.venv\Lib\site-packages\huggingface_hub\file_download.py:144: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\swaya\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/9375 [00:00<?, ?it/s]

(300000, 384)
[[-0.03698691 -0.0214316   0.11477823 ... -0.00515601 -0.00155299
  -0.08958418]
 [-0.05627382 -0.08745369  0.0368     ...  0.06174331 -0.00369593
  -0.05814916]
 [ 0.00012053  0.03061481 -0.02489772 ... -0.01552051  0.04840955
  -0.00649815]
 ...
 [-0.0311013  -0.01838605  0.01456273 ...  0.05102251 -0.05376613
   0.05286839]
 [-0.02998556 -0.00998588  0.00249656 ... -0.07167885 -0.09088012
   0.1055689 ]
 [ 0.03460238 -0.01274594 -0.05576073 ... -0.00344945  0.00224052
   0.00631149]]


In [ ]:
x.shape

(384,)

In [165]:
from sklearn.metrics import mean_squared_error

import numpy as np

average = df['likes'].mean()

rmse = np.sqrt(mean_squared_error(df['likes'], average * np.ones_like(df['likes'])))
print("RMSE:", rmse)

RMSE: 4931.455200077608


In [8]:
import torch
print("CUDA Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Number of GPUs:", torch.cuda.device_count())
    print("Current GPU Device:", torch.cuda.current_device())
    print("GPU Name:", torch.cuda.get_device_name(torch.cuda.current_device()))



CUDA Available: True
Number of GPUs: 1
Current GPU Device: 0
GPU Name: NVIDIA GeForce RTX 4050 Laptop GPU


In [32]:
import torch
import torch.nn as nn

class Like_Predictor(nn.Module):
    def __init__(self, embedding_dimension, numerical_features_count, categorical_features_dict):
        super().__init__()
        # Dense layer for the CLS embedding
        self.bert_embeddings_layer = nn.Sequential(
            nn.Linear(embedding_dimension, 128),
            nn.ReLU()
        )
        # Embedding layers for categorical features
        self.categorical_embeddings_layer = nn.ModuleDict({
            name: nn.Embedding(num_classes, emb_dim)
            for name, (num_classes, emb_dim) in categorical_features_dict.items()
        })
        # Dense for structured numerical features
        self.numerical_features_layer = nn.Sequential(
            nn.Linear(numerical_features_count, 32),
            nn.ReLU()
        )
        # Final classifier
        total_dim = 128 + 32 + sum(emb_dim for _, (_, emb_dim) in categorical_features_dict.items())
        self.final_nn = nn.Sequential(
            nn.Linear(total_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 1)
        )

    def forward(self, bert_embedding, numerical_features, categorical_features):
    
        bert_embeds_layer = self.bert_embeddings_layer(bert_embedding)  # (batch, 128)
        num_layer = self.numerical_features_layer(numerical_features)  # (batch, 32)
        # Process categorical features
        cat_embeds = [
            self.categorical_embeddings_layer[name](categorical_features[name])  # (batch, emb_dim)
            for name in self.categorical_embeddings_layer
        ]
        # Concatenate categorical embeddings
        cat_embeds_layer = torch.cat(cat_embeds, dim=1) if cat_embeds else None

        if cat_embeds_layer is not None:
            all_features_layer = torch.cat([bert_embeds_layer, num_layer, cat_embeds_layer], dim=1)
        else:
            all_features_layer = torch.cat([bert_embeds_layer, num_layer], dim=1)
        out = self.final_nn(all_features_layer)
        return out.squeeze(1)  # (batch,)




In [10]:
bert_embedding = pd.read_csv('embeddings.csv')

In [12]:
numerical_features = pd.concat([
    df[['char_count']],
    usernames_data[['followers', 'following', 'statusesCount', 'favouritesCount', 'mediaCount']]  # columns from usernames_data
], axis=1)


In [26]:
# changing where 'verifiedType' is null to 'None'
usernames_data['verifiedType'] = usernames_data['verifiedType'].fillna('None')

In [27]:
categorical_features = {
    'media_type': df['media_type'].astype('category').cat.codes,
    'hour': df['hour'],
    'verifiedType': usernames_data['verifiedType'].astype('category').cat.codes
}

In [34]:
print((categorical_features['media_type'] < 0).sum())
print((categorical_features['hour'] < 0).sum())
print((categorical_features['verifiedType'] < 0).sum())

0
0
0


In [35]:
print(df['hour'].unique())

[ 0 10 19 11 14 16 22 12  5 15 13 23 20 18  6  3  1  2  7  9 21  8 17  4]


In [36]:
for name, (num_classes, _) in categorical_features_dict.items():
    print(f"{name}: max={categorical_features[name].max()}, num_classes={num_classes}")

media_type: max=2, num_classes=4
hour: max=23, num_classes=24
verifiedType: max=2, num_classes=3


In [39]:
for name in categorical_features:
    print(name, sorted(categorical_features[name].unique()))

media_type [np.int8(0), np.int8(1), np.int8(2)]
hour [np.int32(0), np.int32(1), np.int32(2), np.int32(3), np.int32(4), np.int32(5), np.int32(6), np.int32(7), np.int32(8), np.int32(9), np.int32(10), np.int32(11), np.int32(12), np.int32(13), np.int32(14), np.int32(15), np.int32(16), np.int32(17), np.int32(18), np.int32(19), np.int32(20), np.int32(21), np.int32(22), np.int32(23)]
verifiedType [np.int8(0), np.int8(1), np.int8(2)]


In [37]:
# Example usage:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Configuration
embedding_dimension = 384
numerical_features_count = 6 # char_count, followers, following, statusesCount, favouritesCount, mediaCount
categorical_features_dict = {
    'media_type': (3, 8),     # 4 types, 8-dim embedding
    'hour': (24, 4),     # 24 hours, 4-dim embedding
    'verifiedType': (3, 4)  # 3 types, 4-dim embedding
}

# Model
model = Like_Predictor(embedding_dimension, numerical_features_count, categorical_features_dict).to(device)

# Dummy batch data

batch_size = 30
'''
bert_embedding = torch.randn(batch_size, embedding_dimension).to(device)  # CLS token embedding
numerical_features = torch.randn(batch_size, numerical_features_count).to(device)
categorical_features = {
    'media_type': torch.randint(0, 4, (batch_size,)).to(device),
    'day_of_week': torch.randint(0, 7, (batch_size,)).to(device)
}'''
targets = df['likes']

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.MSELoss()
model.train()
# Training step
for i in range(0, len(bert_embedding), batch_size):
    bert_embedding_batch = torch.tensor(bert_embedding[i:i + batch_size].values, dtype=torch.float32).to(device)
    numerical_features_batch = torch.tensor(numerical_features[i:i + batch_size].values, dtype=torch.float32).to(device)
    categorical_features_batch = {
        name: torch.tensor(categorical_features[name][i:i + batch_size].values, dtype=torch.long).to(device)
        for name in categorical_features
    }
    targets_batch = torch.tensor(targets[i:i + batch_size].values, dtype=torch.float32).to(device)

    optimizer.zero_grad()
    outputs = model(bert_embedding_batch, numerical_features_batch, categorical_features_batch)
    loss = criterion(outputs, targets_batch)
    loss.backward()
    optimizer.step()
    print(f"Batch {i // batch_size + 1}, Loss: {loss.item()}")
# Save the model
joblib.dump(model, 'like_predictor_nn.pkl')


RuntimeError: CUDA error: device-side assert triggered
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


In [41]:
# Example usage:
device = torch.device("cpu")

# Configuration
embedding_dimension = 384
numerical_features_count = 6 # char_count, followers, following, statusesCount, favouritesCount, mediaCount
categorical_features_dict = {
    'media_type': (3, 8),     # 4 types, 8-dim embedding
    'hour': (24, 4),     # 24 hours, 4-dim embedding
    'verifiedType': (3, 4)  # 3 types, 4-dim embedding
}

# Model
model = Like_Predictor(embedding_dimension, numerical_features_count, categorical_features_dict).to(device)

# Dummy batch data

batch_size = 30
'''
bert_embedding = torch.randn(batch_size, embedding_dimension).to(device)  # CLS token embedding
numerical_features = torch.randn(batch_size, numerical_features_count).to(device)
categorical_features = {
    'media_type': torch.randint(0, 4, (batch_size,)).to(device),
    'day_of_week': torch.randint(0, 7, (batch_size,)).to(device)
}'''
targets = df['likes']

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.MSELoss()
model.train()
# Training step
for i in range(0, len(bert_embedding), batch_size):
    bert_embedding_batch = torch.tensor(bert_embedding[i:i + batch_size].values, dtype=torch.float32).to(device)
    numerical_features_batch = torch.tensor(numerical_features[i:i + batch_size].values, dtype=torch.float32).to(device)
    categorical_features_batch = {
        name: torch.tensor(categorical_features[name][i:i + batch_size].values, dtype=torch.long).to(device)
        for name in categorical_features
    }
    targets_batch = torch.tensor(targets[i:i + batch_size].values, dtype=torch.float32).to(device)

    optimizer.zero_grad()
    outputs = model(bert_embedding_batch, numerical_features_batch, categorical_features_batch)
    loss = criterion(outputs, targets_batch)
    loss.backward()
    optimizer.step()
    print(f"Batch {i // batch_size + 1}, Loss: {loss.item()}")
# Save the model
joblib.dump(model, 'like_predictor_nn.pkl')


RuntimeError: CUDA error: device-side assert triggered
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.
